# 03. Feature Processing (Raw)

**Goal**: Prepare raw gameplay features. (Min-Max normalization removed to preserve behavioral semantics).

**Inputs**:
- `data/processed/2_cleaned_telemetry_for_modelling.csv`

**Outputs**:
- `data/processed/3_normalized_telemetry.csv` (Contains RAW data now)

## 1. Setup and Imports

In [1]:
import pandas as pd
import numpy as np
import os

INPUT_PATH = 'data/processed/2_cleaned_telemetry_for_modelling.csv'
OUTPUT_PATH = 'data/processed/3_normalized_telemetry.csv'

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Load Cleaned Data

In [2]:
print("Loading cleaned telemetry data...")
if os.path.exists(INPUT_PATH):
    df = pd.read_csv(INPUT_PATH)
    print(f"Loaded {len(df)} rows. Unique Users: {df['userId'].nunique()}")
    print(df.head())
else:
    raise FileNotFoundError(f"File not found at {INPUT_PATH}")

Loading cleaned telemetry data...
Loaded 2838 rows. Unique Users: 40
                      _id_x                    userId  timestamp  \
0  6974894b48d53c4152cf142a  6974892348d53c4152cf1421        NaN   
1  6974896948d53c4152cf1461  6974892348d53c4152cf1421        NaN   
2  6974898748d53c4152cf148d  6974892348d53c4152cf1421        NaN   
3  697489a548d53c4152cf14b7  6974892348d53c4152cf1421        NaN   
4  697489c348d53c4152cf14e4  6974892348d53c4152cf1421        NaN   

              sessionId  itemsCollected  pickupAttempts  \
0  unreal_1769245003613               0               0   
1  unreal_1769245033617               3               5   
2  unreal_1769245063607               1               2   
3  unreal_1769245093601               3               3   
4  unreal_1769245123579               1              11   

   timeNearInteractables  enemiesHit  damageDone  timeInCombat  ...  \
0                      0           0         0.0           0.0  ...   
1                      7 

## 3. Identify Numeric Features

In [3]:
# Columns to Exclude
EXCLUDE = ['_id', 'userId', 'timestamp', 'session', 'death_count', 'time_since_start']

# Identify Numeric Columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Filter to get Feature Cols
feature_cols = [c for c in numeric_cols if c not in EXCLUDE and not c.startswith('rawJson')]

print(f"Features to Normalize ({len(feature_cols)}):")
print(feature_cols)

Features to Normalize (12):
['itemsCollected', 'pickupAttempts', 'timeNearInteractables', 'enemiesHit', 'damageDone', 'timeInCombat', 'distanceTraveled', 'timeOutOfCombat', 'timeSprinting', 'kills', '__v_x', '__v_y']


## 4. Raw Data Pass-Through
We simply pass the cleaned data through, ensuring NaNs are handled (filled with 0).

In [ ]:
print("Processing raw telemetry (No Normalization)...")
# We just copy the dataframe or fillna
df_norm = df.copy()
df_norm[feature_cols] = df_norm[feature_cols].fillna(0)

# Ensure userId present
if "userId" not in df_norm.columns:
    df_norm["userId"] = df["userId"]

print(df_norm.head())
print("Raw data preparation complete.")

## 5. Verification & Export

In [ ]:
# Safe verification and save
if "userId" in df_norm.columns and not df_norm.empty:
    sample_user = df_norm["userId"].iloc[0]
    print(f"--- Verification (User {sample_user}) ---")
    print("Feature Ranges (Raw Values check - expect > 1):")
    print(df_norm.head())
    print(df_norm[df_norm["userId"] == sample_user][feature_cols].agg(["min", "max"]))
else:
    print("No rows or userId missing; skipping range check.")

df_norm.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved raw data (preserving structure) to: {OUTPUT_PATH}")